# 자연어 처리 3일차 (3) — 줄거리 유사도로 영화 추천 : CountVectorizer · TfidfVectorizer + 코사인

수업 필기 `nlp_실습6` 을 **실행 가능하도록 정리 · 개선한 노트**입니다.
(교재 `자연어처리` 교재v1 **p.63~65** — CountVectorizer / TfidfVectorizer 와 코사인 유사도를 이용한 추천 시스템 구현 실습)

1. `movies_metadata.csv` 에서 줄거리(`overview`) · 제목(`title`)
2. 줄거리 → `CountVectorizer`(또는 `TfidfVectorizer`) 행렬
3. 좋아하는 영화 제목 입력 → 그 영화 줄거리와 **코사인 유사도가 높은** 영화 추천

- 원본에서 고친 부분은 **📝 고친 부분** 으로 적었습니다. 앞 노트 : [`bow_count_tfidf.ipynb`](./bow_count_tfidf.ipynb) · [`text_similarity_cosine.ipynb`](./text_similarity_cosine.ipynb)

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `pandas` | csv 읽기 · 전처리 | Colab 기본 포함 |
| `scikit-learn` | `CountVectorizer` · `TfidfVectorizer` · `cosine_similarity` | Colab 기본 포함 |
| `scipy` | 희소 행렬 (`scikit-learn` 과 함께 설치됨) | — |

**데이터** — Kaggle [The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset) 의 `movies_metadata.csv` (34MB, 45,466행 · 24열). 둘 중 하나를 노트북과 **같은 폴더**에 둔다.
- 원본 `movies_metadata.csv` — Kaggle 에서 받기 (Colab : 왼쪽 📁 에 업로드)
- 또는 이 폴더의 `data/movies_overview.csv.gz` (7MB) — 원본에서 이 노트가 쓰는 5개 열(`id · title · overview · release_date · status`)만 뽑아 압축한 것. 노트가 자동으로 찾는다

**실행 순서** : 맨 위부터 **[런타임] → [모두 실행]**. 원본처럼 `.toarray()` 로 바꾸지 않으므로 **메모리 2GB 이하**에서도 4만 편 전체가 돌아간다 (CPU 로 수십 초).

In [1]:
# ① 설치 (로컬에서 처음 한 번 — Colab 은 기본 포함)
# !pip install -q pandas scikit-learn

In [2]:
import os
import numpy as np
import pandas as pd

for path in ['movies_metadata.csv', 'data/movies_overview.csv.gz', 'movies_overview.csv.gz']:
    if os.path.exists(path):
        break
else:
    raise FileNotFoundError('movies_metadata.csv (또는 data/movies_overview.csv.gz) 를 노트북 폴더에 두세요')

df = pd.read_csv(path, low_memory=False)          # 섞인 자료형 열 → DtypeWarning 없이 한 번에 읽기
print(path, df.shape)
df[['title', 'overview', 'release_date']].head(3)

movies_metadata.csv (45466, 24)


,title,overview,release_date
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...",1995-10-30
1,Jumanji,When siblings Judy and Peter discover an encha...,1995-12-15
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,1995-12-22


> **📝 고친 부분 (원본 셀 1~2)**
> - `drive.mount(...)` 후 `'/content/movies_metadata.csv'` 를 읽었다 → 드라이브가 아닌 Colab 임시 폴더라 **마운트가 필요 없다**. 파일 위치를 한 곳(`path`)으로
> - `DtypeWarning: Columns (10) have mixed types` → `low_memory=False` (10번 열 `id` 에 날짜 같은 깨진 값이 섞여 있다 — 아래 1절)
> - 오타 `임포` · `임퐅트` · `임배딩` · `켤과` · `커럼` · `vecab` → 임포트 · 임베딩 · 결과 · 컬럼 · **vocab**(vocabulary)

## 1. 데이터 점검 — 교재 '2017년 7월 이전 개봉 영화 45,466편' 확인

In [3]:
id_num = pd.to_numeric(df['id'], errors='coerce')
date = pd.to_datetime(df['release_date'], errors='coerce')
print('전체 행                       :', len(df))
print('id 가 숫자가 아닌 깨진 행     :', id_num.isna().sum(), df.loc[id_num.isna(), 'id'].tolist())
print('같은 id 가 또 나온 중복 행    :', id_num[id_num.notna()].duplicated().sum())
print('개봉일 2017-08-01 이후        :', (date >= '2017-08-01').sum(), '| 가장 늦은 날짜', date.max().date())
print('\nstatus :', df['status'].value_counts().to_dict())

전체 행                       : 45466
id 가 숫자가 아닌 깨진 행     : 3 ['1997-08-20', '2012-09-29', '2014-01-01']
같은 id 가 또 나온 중복 행    : 30
개봉일 2017-08-01 이후        : 40 | 가장 늦은 날짜 2020-12-16

status : {'Released': 45014, 'Rumored': 230, 'Post Production': 98, 'In Production': 20, 'Planned': 15, 'Canceled': 2}


> **📝 고친 부분 (교재 p.63 · p.65 '2017년 7월 또는 그 이전에 개봉된 영화 45,466편')**
> - 45,466 은 **행 수**다. 그중 **3행은 깨진 행**(id 자리에 날짜), **30행은 같은 영화의 중복** → 실제 영화는 약 45,430편
> - 개봉일이 **2017년 8월 이후(최대 2020-12)** 인 행이 40개, `Rumored` · `Planned` · `In Production` 같은 **미개봉** 영화도 들어 있다 — '2017년 7월 이전 개봉' 은 대부분일 뿐 전부는 아니다

## 2. 전처리 — `overview` · `title`

In [4]:
df1 = df[['title', 'overview', 'release_date']]
print('결측치 :', df1[['overview', 'title']].isnull().sum().to_dict())

movies = df1.dropna(subset=['overview', 'title']).reset_index(drop=True)
print('결측치 제거 후 :', len(movies), '편')

# 줄거리가 사실상 비어 있는 자리표시 문구
placeholder = movies['overview'].str.strip().str.lower().isin(['', 'no overview found.', 'no overview', 'no movie overview available.'])
print('자리표시 줄거리 :', placeholder.sum(), movies.loc[placeholder, 'overview'].value_counts().to_dict())

결측치 : {'overview': 954, 'title': 6}
결측치 제거 후 : 44506 편
자리표시 줄거리 : 149 {'No overview found.': 133, 'No Overview': 7, ' ': 5, 'No movie overview available.': 3, 'No overview': 1}


In [5]:
movies = movies[~placeholder].reset_index(drop=True)
movies['year'] = pd.to_datetime(movies['release_date'], errors='coerce').dt.year.astype('Int64')
print('최종 :', len(movies), '편')
print('같은 제목이 2번 이상인 영화 :', movies['title'].duplicated(keep=False).sum(), '행  예)',
      movies.loc[movies['title'] == 'Hamlet', ['title', 'year']].values.tolist()[:4])

최종 : 44357 편
같은 제목이 2번 이상인 영화 : 5423 행  예) [['Hamlet', 1996], ['Hamlet', 1948], ['Hamlet', 2000], ['Hamlet', 1990]]


> **📝 고친 부분 (원본 셀 5)**
> - 주석 '결측치 제거 후 원본 df1 에 반영' → `dropna` 는 **새 표 `df2`** 를 만든다(df1 은 그대로). 확인 코드도 df1 을 다시 세고 있어 주석 처리돼 있었다
> - `No overview found.` 같은 **자리표시 문구 149개**가 줄거리로 남아 있었다 → 이 영화들은 서로 유사도 **1.0** 이 되어 엉뚱하게 추천된다. 제거
> - 같은 제목이 여러 편(예 : `Hamlet`)이라 **제목만으로는 한 편을 못 고른다** → 개봉 연도를 함께 보여 준다(4절)

## 3. CountVectorizer 행렬 — 희소 행렬 그대로 (교재 p.63~64)

In [6]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words='english')          # 영어 불용어(the, a, of …) 제거
X_count = cv.fit_transform(movies['overview'])      # fit + transform 한 번에
vocab = cv.vocabulary_

print('행렬 모양 :', X_count.shape, '| 0 아닌 칸 :', X_count.nnz)
print('단어 사전 일부 :', dict(list(vocab.items())[:8]))

dense_gb = X_count.shape[0] * X_count.shape[1] * 8 / 1e9
sparse_mb = (X_count.data.nbytes + X_count.indices.nbytes + X_count.indptr.nbytes) / 1e6
print(f'.toarray() 로 바꾸면 : {dense_gb:,.1f} GB   vs   희소 행렬 : {sparse_mb:,.1f} MB')
print(f'전체 유사도 행렬 ({len(movies):,}²) : {len(movies) ** 2 * 8 / 1e9:,.1f} GB')

행렬 모양 : (44357, 75827) | 0 아닌 칸 : 1210689
단어 사전 일부 : {'led': 38088, 'woody': 73468, 'andy': 3159, 'toys': 67874, 'live': 39012, 'happily': 28729, 'room': 56872, 'birthday': 7491}
.toarray() 로 바꾸면 : 26.9 GB   vs   희소 행렬 : 14.7 MB
전체 유사도 행렬 (44,357²) : 15.7 GB


> **📝 고친 부분 (원본 셀 8~11)**
> - `df2[0:9999]` + 주석 '1만 개(0~9999)' → 슬라이스 끝은 포함하지 않아 **9,999편**. 1만 편이면 `[:10000]`
> - 1만 편으로 줄인 이유는 `.toarray()` 와 **전체 × 전체 유사도 행렬** 때문이다. 원본 크기(9,999 × 32,382)만 해도 정수 행렬 약 **2.6GB** + 유사도 표 0.8GB × 2(`cos_sim`, `df_sim`)
> - 고친 방법 : 행렬은 **희소 행렬 그대로** 두고, 유사도는 **입력한 영화 한 편 vs 전체**만 계산 → 4만 편 전체로도 가볍다

## 4. 추천 함수

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

title_idx = pd.Series(movies.index, index=movies['title'])     # 제목 → 행 번호 (같은 제목이면 여러 개)

import difflib

def recommend(title, k=10, X=X_count, year=None, quiet=False):
    """title 과 줄거리가 비슷한 영화 k 편 (자기 자신 제외)"""
    if title not in title_idx.index:
        cand = difflib.get_close_matches(title, movies['title'].unique(), n=5, cutoff=0.6)
        print(f"'{title}' 은(는) 없는 제목입니다. 비슷한 제목 : {list(cand)}")
        return pd.DataFrame()
    rows = np.atleast_1d(title_idx[title])
    if year is not None:
        rows = [r for r in rows if movies.at[r, 'year'] == year] or rows
    if len(rows) > 1 and not quiet:
        print(f"'{title}' 이(가) {len(rows)}편 — {movies.loc[rows, 'year'].tolist()} 중 첫 번째 기준 (year= 로 고르기)")
    i = rows[0]
    sims = cosine_similarity(X[i], X).ravel()      # 한 편 vs 전체 (1 × N)
    sims[i] = -1                                    # 자기 자신은 위치로 제외
    top = np.argsort(-sims)[:k]
    return pd.DataFrame({'title': movies['title'].values[top], 'year': movies['year'].values[top],
                         'similarity': sims[top].round(3)}, index=range(1, k + 1))

recommend('Toy Story')

,title,year,similarity
1,Toy Story 3,2010,0.436
2,Toy Story 2,1999,0.412
3,The 40 Year Old Virgin,2005,0.293
4,The Champ,1931,0.245
5,Andy Hardy's Blonde Trouble,1944,0.244
6,Andy Kaufman Plays Carnegie Hall,1980,0.231
7,Andy Peters: Exclamation Mark Question Point,2015,0.216
8,Superstar: The Life and Times of Andy Warhol,1990,0.214
9,Small Fry,2011,0.208
10,Wabash Avenue,1950,0.177


In [8]:
recommend('GoldenEye')

,title,year,similarity
1,Licence to Kill,1989,0.241
2,Never Say Never Again,1983,0.197
3,You Only Live Twice,1967,0.195
4,Casino Royale,1967,0.195
5,Octopussy,1983,0.186
6,Bunraku,2010,0.182
7,Live and Let Die,1973,0.182
8,Phantasm,1979,0.182
9,Dr. No,1962,0.174
10,The Tall Target,1951,0.167


In [9]:
recommend('Toy Storyy')        # 없는 제목 → None 대신 빈 표 + 비슷한 제목 안내

'Toy Storyy' 은(는) 없는 제목입니다. 비슷한 제목 : ['Toy Story', 'Toy Story 3', 'Toy Story 2', 'Tokyo Story', 'True Story']


""


In [10]:
recommend('Hamlet', k=5, year=1996)       # 같은 제목 여러 편 → 연도로 고르기

,title,year,similarity
1,Hamlet,1990,0.400
2,Hamlet,1980,0.355
3,Lily Sometimes,2010,0.314
4,Stir of Echoes: The Homecoming,2007,0.294
5,Stay,2013,0.294


> **📝 고친 부분 (원본 셀 12~14 `top_n_func`)**
> - `sort_values(...)[1:k+1]` 은 '1등은 자기 자신' 이라고 **가정**한다 → 줄거리가 똑같은 다른 영화(중복 행)가 1.0 으로 같이 있으면 자기 자신이 2등으로 밀려 결과에 섞인다. **자기 위치를 −1 로** 지우고 정렬
> - `df_sim[title]` 은 같은 제목이 둘 이상이면 Series 가 아니라 **표(DataFrame)** 가 나와 `sort_values` 가 실패한다 → 행 번호로 다루고 `year=` 로 고른다
> - 없는 제목이면 `print` 만 하고 **`None` 을 돌려줘** 바로 아래 `print(top_10)` 이 `None` 을 찍었다 → 빈 표 + 비슷한 제목 안내
> - 함수 인자 순서 `top_n_func(k, title)` → `recommend(title, k=10)` (제목이 먼저, 개수는 기본값)

## 5. 원본 결과 재현 — 앞 9,999편만 썼을 때

In [11]:
d = df[['overview', 'title']].dropna().reset_index(drop=True)[0:9999]      # 원본과 똑같이
cv9 = CountVectorizer(stop_words='english')
X9 = cv9.fit_transform(d['overview'])
s = cosine_similarity(X9[d.index[d['title'] == 'Toy Story'][0]], X9).ravel()
print('원본 크기 :', X9.shape)
pd.Series(s, index=d['title']).sort_values(ascending=False)[1:6].round(6)

원본 크기 : (9999, 32382)


title
Toy Story 2         0.412393
The Champ           0.245402
Rivers and Tides    0.142857
Man on the Moon     0.142857
Class of 1984       0.139047
dtype: float64

원본 출력(`Toy Story 2  0.412393` · `The Champ  0.245402` …)과 **같다** (0.142857 동점인 두 편은 순서가 바뀔 수 있다). 전체 4만 편으로 넓히면 후보가 4배 이상이라 순위가 바뀐다(4절).

## 6. TfidfVectorizer 로 바꾸기 (교재 p.65)

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
X_tfidf = tfidf.fit_transform(movies['overview'])
print('TF-IDF 행렬 :', X_tfidf.shape)

def side_by_side(title, k=10):
    a = recommend(title, k, X_count)
    b = recommend(title, k, X_tfidf, quiet=True)
    return pd.DataFrame({'Count 추천': a['title'], 'Count 유사도': a['similarity'],
                         'TF-IDF 추천': b['title'], 'TF-IDF 유사도': b['similarity']})

side_by_side('Toy Story')

TF-IDF 행렬 : (44357, 75827)


,Count 추천,Count 유사도,TF-IDF 추천,TF-IDF 유사도
1,Toy Story 3,0.436,Toy Story 3,0.532
2,Toy Story 2,0.412,Toy Story 2,0.472
3,The 40 Year Old Virgin,0.293,The 40 Year Old Virgin,0.275
4,The Champ,0.245,Small Fry,0.273
5,Andy Hardy's Blonde Trouble,0.244,Andy Hardy's Blonde Trouble,0.235
6,Andy Kaufman Plays Carnegie Hall,0.231,Hot Splash,0.224
7,Andy Peters: Exclamation Mark Question Point,0.216,Andy Kaufman Plays Carnegie Hall,0.218
8,Superstar: The Life and Times of Andy Warhol,0.214,Superstar: The Life and Times of Andy Warhol,0.216
9,Small Fry,0.208,Andy Peters: Exclamation Mark Question Point,0.202
10,Wabash Avenue,0.177,The Champ,0.199


In [13]:
side_by_side('GoldenEye')

,Count 추천,Count 유사도,TF-IDF 추천,TF-IDF 유사도
1,Licence to Kill,0.241,Licence to Kill,0.154
2,Never Say Never Again,0.197,Dream Work,0.141
3,You Only Live Twice,0.195,Live and Let Die,0.138
4,Casino Royale,0.195,Octopussy,0.134
5,Octopussy,0.186,You Only Live Twice,0.134
6,Bunraku,0.182,Doctor X,0.132
7,Live and Let Die,0.182,Johnny Stool Pigeon,0.131
8,Phantasm,0.182,Never Say Never Again,0.131
9,Dr. No,0.174,Casino Royale,0.123
10,The Tall Target,0.167,The Way of the Dragon,0.122


In [14]:
# 두 방법이 얼마나 같은 영화를 추천하나 (상위 10편 겹침)
for t in ['Toy Story', 'GoldenEye', 'The Dark Knight', 'Titanic', 'Finding Nemo']:
    a = set(recommend(t, 10, X_count, quiet=True)['title']); b = set(recommend(t, 10, X_tfidf, quiet=True)['title'])
    print(f'{t:16s} 겹침 {len(a & b):2d}/10')

Toy Story        겹침  9/10
GoldenEye        겹침  6/10
The Dark Knight  겹침  5/10


Titanic          겹침  5/10
Finding Nemo     겹침  1/10


In [15]:
# TF-IDF 가 줄거리에서 무게를 준 단어 상위 8개
feat = tfidf.get_feature_names_out()
for t in ['Toy Story', 'Finding Nemo']:
    r = X_tfidf[title_idx[t]].toarray().ravel()
    print(f'{t:13s}', [(feat[j], round(float(r[j]), 3)) for j in np.argsort(-r)[:8]])

Toy Story     [('buzz', 0.503), ('woody', 0.481), ('andy', 0.412), ('lightyear', 0.206), ('toys', 0.149), ('aside', 0.147), ('afraid', 0.141), ('differences', 0.135)]
Finding Nemo  [('nemo', 0.389), ('clownfish', 0.228), ('seagulls', 0.219), ('dory', 0.213), ('worrisome', 0.209), ('forgetful', 0.209), ('marlin', 0.209), ('jellyfish', 0.202)]


- TF-IDF 는 `life` · `young` · `new` 처럼 수많은 줄거리에 나오는 단어의 무게를 줄이고, `buzz` · `woody` · `nemo` 같은 **드문 고유명사**에 무게를 싣는다 (위 셀).
- 잘 맞힌 것 : `Toy Story 2 · 3` 과 `Small Fry`(2011, 버즈가 나오는 토이 스토리 단편)
- 함정 : `andy`(장난감 주인) · `woody` 라는 **이름만 같은** 영화가 섞인다 — `The 40 Year Old Virgin`(주인공 Andy) · `The Champ`(1931, 권투 선수 Andy) · `Andy Warhol` · `Andy Kaufman` 다큐 · `Hot Splash`(주인공 Woody). BoW 는 뜻이 아니라 **글자가 같은지**만 본다 → 밀집 임베딩이 필요한 이유 ([`text_similarity_cosine.ipynb`](./text_similarity_cosine.ipynb) §5)
- TF-IDF 행은 이미 길이 1 이라 `cosine_similarity` 대신 **내적**(`linear_kernel`)만 해도 같은 값이다.

## 정리

| 단계 | 원본 `nlp_실습6` | 고친 노트 |
|---|---|---|
| 데이터 | 앞 9,999편 | 결측 · 자리표시 줄거리를 뺀 전체 |
| 행렬 | `.toarray()` (GB 단위) | 희소 행렬 그대로 (수십 MB) |
| 유사도 | 전체 × 전체 표 `df_sim` | 입력 영화 1편 × 전체 |
| 자기 자신 제외 | `[1:n+1]` (1등 = 자신 가정) | 자기 위치를 −1 |
| 같은 제목 · 없는 제목 | 오류 / `None` | `year=` 선택 · 비슷한 제목 안내 |
| 모델 | CountVectorizer | Count · TF-IDF 비교 |